# Projet Data Analyst — Sales & Business Performance Cameroon 🇨🇲

## Structure du workflow

**Base SQL → SQL → Python/Pandas → Visualisation → Recommandations métier → Power BI**

Ce notebook montre explicitement **où SQL intervient** et où Python prend le relais.

**Jeu de données : 5 000 transactions synthétiques**, cohérent avec la base SQLite, le fichier Excel et le projet Power BI de ce repository.


# 0. Connexion à SQLite

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# Repository structure:
# notebooks/Projet_01_SQL_Python.ipynb
# data/database/sales_performance_cameroon.db
DB_PATH = Path("../data/database/sales_performance_cameroon.db")

if not DB_PATH.exists():
    raise FileNotFoundError(
        "Base SQLite introuvable. Lancez ce notebook depuis le dossier notebooks "
        "du repository ou vérifiez data/database/sales_performance_cameroon.db."
    )

con = sqlite3.connect(DB_PATH)
print("Connexion réussie :", DB_PATH.resolve())


# PARTIE 1 — SQL

Ici, **SQL fait l'analyse**. Python sert seulement à envoyer la requête et afficher le résultat avec `pd.read_sql_query()`.


## 1.1 SELECT — voir les données

In [ ]:
query = """
SELECT *
FROM fact_sales
LIMIT 10;
"""
pd.read_sql_query(query, con)

## 1.2 KPI — COUNT, SUM, AVG

In [ ]:
query = """
SELECT
 COUNT(*) AS transactions,
 ROUND(SUM(ca_net_fcfa),0) AS chiffre_affaires_fcfa,
 ROUND(SUM(marge_brute_fcfa),0) AS marge_brute_fcfa,
 ROUND(100.0*SUM(marge_brute_fcfa)/SUM(ca_net_fcfa),2) AS taux_marge_pct,
 ROUND(AVG(ca_net_fcfa),0) AS panier_moyen_fcfa
FROM fact_sales;
"""
kpi_sql = pd.read_sql_query(query, con)
kpi_sql

## 1.3 WHERE — filtrer les ventes 2026 avec remise

In [ ]:
query = """
SELECT transaction_id,date_vente,ca_net_fcfa,taux_remise
FROM fact_sales
WHERE date_vente >= '2026-01-01'
  AND taux_remise > 0
ORDER BY ca_net_fcfa DESC
LIMIT 15;
"""
pd.read_sql_query(query, con)

## 1.4 GROUP BY — CA par canal

In [ ]:
query = """
SELECT canal,
       COUNT(*) AS transactions,
       ROUND(SUM(ca_net_fcfa),0) AS ca_fcfa
FROM fact_sales
GROUP BY canal
ORDER BY ca_fcfa DESC;
"""
canal_sql = pd.read_sql_query(query, con)
canal_sql

## 1.5 JOIN — CA par agence

In [ ]:
query = """
SELECT s.ville,
       COUNT(*) AS transactions,
       ROUND(SUM(f.ca_net_fcfa),0) AS ca_fcfa,
       ROUND(SUM(f.marge_brute_fcfa),0) AS marge_fcfa
FROM fact_sales f
JOIN dim_store s
  ON f.store_id = s.store_id
GROUP BY s.ville
ORDER BY ca_fcfa DESC;
"""
ville_sql = pd.read_sql_query(query, con)
ville_sql

## 1.6 JOIN — Top produits

In [ ]:
query = """
SELECT p.produit,p.categorie,
       ROUND(SUM(f.ca_net_fcfa),0) AS ca_fcfa,
       ROUND(SUM(f.marge_brute_fcfa),0) AS marge_fcfa
FROM fact_sales f
JOIN dim_product p
  ON f.product_id = p.product_id
GROUP BY p.product_id,p.produit,p.categorie
ORDER BY ca_fcfa DESC
LIMIT 10;
"""
top_produits_sql = pd.read_sql_query(query, con)
top_produits_sql

## 1.7 CTE + LAG — croissance mensuelle

In [ ]:
query = """
WITH ca_mensuel AS (
 SELECT SUBSTR(date_vente,1,7) AS mois,
        SUM(ca_net_fcfa) AS ca
 FROM fact_sales
 GROUP BY mois
),
evolution AS (
 SELECT mois,ca,
        LAG(ca) OVER(ORDER BY mois) AS ca_prec
 FROM ca_mensuel
)
SELECT mois,
       ROUND(ca,0) AS ca_fcfa,
       ROUND(100.0*(ca-ca_prec)/ca_prec,2) AS croissance_mom_pct
FROM evolution
ORDER BY mois;
"""
croissance_sql = pd.read_sql_query(query, con)
croissance_sql.tail(12)

## 1.8 Vue SQL pour la suite

La vue `vw_sales_analysis` regroupe les tables grâce aux `JOIN`.  
Nous l'utilisons ensuite comme table analytique pour Python.


In [ ]:
df = pd.read_sql_query(
    "SELECT * FROM vw_sales_analysis",
    con,
    parse_dates=["date_vente"]
)
df.head()

# PARTIE 2 — Python / Pandas

Ici, Python prend le relais pour le contrôle qualité, les transformations et les analyses complémentaires.


## 2.1 Contrôle qualité

In [ ]:
print("Dimensions :", df.shape)
print("Doublons :", df["transaction_id"].duplicated().sum())
print("Valeurs manquantes :", int(df.isna().sum().sum()))
print("Période :", df["date_vente"].min().date(), "→", df["date_vente"].max().date())

## 2.2 Même analyse en Pandas

Nous avons déjà calculé le CA par ville en SQL. Voici l'équivalent Pandas.


In [ ]:
ville_python = (
    df.groupby("ville", as_index=False)
      .agg(
          chiffre_affaires_fcfa=("ca_net_fcfa","sum"),
          marge_brute_fcfa=("marge_brute_fcfa","sum"),
          transactions=("transaction_id","nunique")
      )
      .sort_values("chiffre_affaires_fcfa", ascending=False)
)
ville_python

## 2.3 Segments clients

In [ ]:
segments_python = (
    df.groupby("segment_client", as_index=False)
      .agg(
          chiffre_affaires_fcfa=("ca_net_fcfa","sum"),
          marge_brute_fcfa=("marge_brute_fcfa","sum"),
          panier_moyen_fcfa=("ca_net_fcfa","mean")
      )
)
segments_python

## 2.4 Analyse des remises

In [ ]:
df["groupe_remise"] = df["taux_remise"].gt(0).map({True:"Avec remise", False:"Sans remise"})
remises_python = (
    df.groupby("groupe_remise", as_index=False)
      .agg(
          panier_moyen_fcfa=("ca_net_fcfa","mean"),
          quantite_moyenne=("quantite","mean"),
          ca_fcfa=("ca_net_fcfa","sum"),
          marge_fcfa=("marge_brute_fcfa","sum")
      )
)
remises_python["taux_marge"] = remises_python["marge_fcfa"] / remises_python["ca_fcfa"]
remises_python

## 2.5 Part du Mobile Money

In [ ]:
mask = df["mode_paiement"].isin(["MTN MoMo","Orange Money"])
part_mobile_money = df.loc[mask,"ca_net_fcfa"].sum() / df["ca_net_fcfa"].sum()
print(f"Part du CA payée par Mobile Money : {part_mobile_money:.2%}")

# PARTIE 3 — Visualisation

Chaque graphique répond à une question métier.


## 3.1 CA par agence

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(ville_python["ville"], ville_python["chiffre_affaires_fcfa"]/1_000_000)
plt.title("Chiffre d'affaires par agence")
plt.xlabel("Ville")
plt.ylabel("CA (millions FCFA)")
plt.tight_layout()
plt.show()

## 3.2 Évolution mensuelle

In [ ]:
mensuel = (
    df.assign(mois=df["date_vente"].dt.to_period("M").astype(str))
      .groupby("mois", as_index=False)
      .agg(ca_fcfa=("ca_net_fcfa","sum"))
)
plt.figure(figsize=(12,5))
plt.plot(mensuel["mois"],mensuel["ca_fcfa"]/1_000_000,marker="o")
plt.title("Évolution mensuelle du chiffre d'affaires")
plt.xlabel("Mois")
plt.ylabel("CA (millions FCFA)")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

# PARTIE 4 — Recommandations métier

À partir des analyses :
- comparer les agences avec **CA + marge + panier moyen** ;
- renforcer les stocks avant les périodes saisonnières fortes ;
- développer les segments PME et revendeurs ;
- traiter le Mobile Money comme un mode de paiement stratégique ;
- surveiller les remises qui dégradent la marge ;
- évaluer les commerciaux sur la rentabilité créée, pas seulement sur le CA.


# PARTIE 5 — Power BI

La prochaine étape sera la restitution décisionnelle.

**KPI :** CA, marge, taux de marge, transactions, panier moyen.

**Visuels :** CA par ville, catégories, top produits, évolution mensuelle, segments clients, paiements, commerciaux.

**Filtres :** année, ville, catégorie, segment, canal, mode de paiement.


# Conclusion

Tu dois pouvoir expliquer le workflow ainsi :

> **SQL** : extraction, filtres, jointures et agrégations.  
> **Python/Pandas** : contrôle qualité et analyses complémentaires.  
> **Matplotlib** : visualisation exploratoire.  
> **Power BI** : dashboard décisionnel.


In [ ]:
con.close()
print("Analyse terminée.")